# 5. Análise de concordância (Fleiss' Kappa)
Calcula a concordância entre avaliadores para cada item (sample_id, smell).

### 5.1 Preparação da base para análise de concordância

Carrega a base original e a base filtrada com links válidos

Em seguida, realiza a padronização dos nomes das colunas e dos valores textuais, além da remoção de duplicatas de avaliações (mesmo revisor avaliando o mesmo `sample_id` e `smell`).

In [1]:
import pandas as pd

# Prepara df_original_200 caso este notebook seja executado de forma independente
df_original = pd.read_excel("MLCQCodeSmellSamples.xlsx")
df_200 = pd.read_excel("MLCQ_status_200.xlsx")

df_original.columns = df_original.columns.str.strip()
df_200.columns = df_200.columns.str.strip()

LINK_COL = "link"
df_original[LINK_COL] = df_original[LINK_COL].astype(str).str.strip()
df_200[LINK_COL] = df_200[LINK_COL].astype(str).str.strip()

links_validos = set(df_200[LINK_COL].dropna().unique())
df_original_200 = df_original[df_original[LINK_COL].isin(links_validos)].copy()
df_original_200 = df_original_200.drop_duplicates(subset=["sample_id", "smell", "reviewer_id"])
df_original_200["severity"] = df_original_200["severity"].astype(str).str.strip().str.lower()
df_original_200["smell"] = df_original_200["smell"].astype(str).str.strip().str.lower()
df_original_200["type"] = df_original_200["type"].astype(str).str.strip().str.lower()

### 5.2 Construção da variável binária de presença de code smell

Cria a variável `tem_smell`, que indica a presença (1) ou ausência (0) de code smell com base no campo `severity`.

Valores diferentes de "none" são considerados como presença de smell, enquanto "none" indica ausência.

In [2]:
df_original_200["tem_smell"] = (df_original_200["severity"] != "none").astype(int)
print(df_original_200.head(5))

    id  reviewer_id  sample_id         smell  severity  \
0  526            6    5771277  feature envy      none   
1  527            6    5771277   long method      none   
2  528            6    5786929          blob  critical   
3  529            6    5786929    data class  critical   
4  530            6    5788107  feature envy      none   

             review_timestamp      type  \
0  2019-03-27 10:34:53.041496  function   
1  2019-03-27 10:34:53.042443  function   
2  2019-03-27 10:37:38.107923     class   
3  2019-03-27 10:37:38.109068     class   
4  2019-03-27 10:37:49.627100  function   

                                           code_name  \
0  org.apache.syncope.client.ui.commons.ConnIdSpe...   
1  org.apache.syncope.client.ui.commons.ConnIdSpe...   
2  org.apache.tez.runtime.library.common.writers....   
3  org.apache.tez.runtime.library.common.writers....   
4  org.apache.tika.parser.ocr.TesseractOCRConfig#...   

                          repository  \
0  git@github.c

### 5.3 Construção da matriz de avaliações por instância

Agrupa os dados por (`sample_id`, `smell`) e contabiliza, para cada instância, o número de votos indicando presença e ausência de code smell.

A estrutura resultante é uma matriz onde cada linha representa uma instância avaliada, e as colunas indicam a quantidade de votos para cada categoria (0 = ausência, 1 = presença).

In [3]:
matriz_kappa = (
    df_original_200.groupby(["sample_id", "smell"])["tem_smell"]
    .value_counts()
    .unstack(fill_value=0)
)
print(matriz_kappa.head())

tem_smell               0  1
sample_id smell             
3698323   blob          1  0
          data class    1  0
3698602   feature envy  1  0
          long method   1  0
3698665   feature envy  1  0


### 5.4 Padronização da matriz para cálculo do coeficiente Kappa

Garante que a matriz contenha explicitamente as duas categorias possíveis (0 e 1), mesmo que uma delas não esteja presente em determinada instância.

Essa padronização é necessária para a aplicação correta do coeficiente de Fleiss' Kappa, que exige consistência no número de categorias avaliadas.

In [4]:
if 0 not in matriz_kappa.columns:
    matriz_kappa[0] = 0
if 1 not in matriz_kappa.columns:
    matriz_kappa[1] = 0

matriz_kappa = matriz_kappa[[0, 1]]

In [5]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


### 5.5 Interpretação da concordância interavaliador

Os valores obtidos para o coeficiente de Fleiss' Kappa indicam o nível de concordância entre os revisores além do esperado pelo acaso.

Observa-se que:

- há um grande número de instâncias avaliadas por apenas um revisor, impossibilitando o cálculo de concordância nesses casos
- para instâncias com múltiplos avaliadores, os valores de Kappa situam-se em níveis baixos a moderados
- essa baixa concordância pode ser atribuída à natureza subjetiva da identificação de code smells e à predominância de avaliações negativas (ausência de smell)

Esses resultados sugerem que a identificação de code smells apresenta variabilidade entre avaliadores, justificando o uso de estratégias de agregação, como majority vote, para definição do rótulo final (ground truth).

In [6]:
from statsmodels.stats.inter_rater import fleiss_kappa

# Matriz com número de votos 0 e 1 por item
matriz_kappa = (
    df_original_200.groupby(["sample_id", "smell"])["tem_smell"]
    .value_counts()
    .unstack(fill_value=0)
)

if 0 not in matriz_kappa.columns:
    matriz_kappa[0] = 0
if 1 not in matriz_kappa.columns:
    matriz_kappa[1] = 0

matriz_kappa = matriz_kappa[[0, 1]]

# Ver quantos avaliadores há por item
contagem_avaliadores = matriz_kappa.sum(axis=1)
print(contagem_avaliadores.value_counts().sort_index())

1    6484
2     372
3    1289
4     483
5      83
Name: count, dtype: int64


### 5.6 Cálculo do coeficiente de Fleiss' Kappa por número de avaliadores

O coeficiente de Fleiss' Kappa foi calculado separadamente para subconjuntos de instâncias com o mesmo número de avaliadores, uma vez que o método requer um número fixo de julgadores por item.

Para cada valor de `n` avaliadores, foi extraído o subconjunto correspondente e calculado o Kappa com base na distribuição de votos de presença (1) e ausência (0) de code smell.

In [7]:
for n in sorted(contagem_avaliadores.unique()):
    subset = matriz_kappa[contagem_avaliadores == n]
    
    if len(subset) > 1:
        kappa = fleiss_kappa(subset.values)
        print(f"{n} avaliadores: {len(subset)} itens | Fleiss' Kappa = {kappa}")

1 avaliadores: 6484 itens | Fleiss' Kappa = nan
2 avaliadores: 372 itens | Fleiss' Kappa = 0.9892445138346778
3 avaliadores: 1289 itens | Fleiss' Kappa = 0.28523899301319705
4 avaliadores: 483 itens | Fleiss' Kappa = 0.3032408986792728
5 avaliadores: 83 itens | Fleiss' Kappa = 0.35631151746979545


c:\Users\Oscar Neto\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\stats\inter_rater.py:258: RuntimeWarning: invalid value encountered in divide
  p_rat = (table2.sum(1) - n_rat) / (n_rat * (n_rat - 1.))


### 5.7 Distribuição das classes de presença de code smell

Apresenta a distribuição global da variável `tem_smell`, indicando a proporção de avaliações que identificam presença ou ausência de code smells.

In [8]:
df_original_200["tem_smell"].value_counts(normalize=True)

tem_smell
0    0.779348
1    0.220652
Name: proportion, dtype: float64

### 5.8 Distribuição da presença de code smells por categoria

Apresenta a distribuição da variável `tem_smell` para cada tipo de code smell (`blob`, `data class`, `feature envy`, `long method`).

In [9]:
df_original_200.groupby("smell")["tem_smell"].value_counts(normalize=True)

smell         tem_smell
blob          0            0.762384
              1            0.237616
data class    0            0.733950
              1            0.266050
feature envy  0            0.864295
              1            0.135705
long method   0            0.769961
              1            0.230039
Name: proportion, dtype: float64

## 6. Construção do ground truth

### 6.1 Agregação das avaliações por instância

A construção do ground truth foi realizada a partir da agregação das avaliações individuais dos revisores.

As avaliações foram agrupadas por (`sample_id`, `smell`), permitindo consolidar, para cada instância, o número total de avaliadores e a quantidade de votos indicando presença ou ausência de code smell.

Essa abordagem transforma a base original, composta por avaliações individuais, em uma estrutura consolidada adequada para definição de rótulos finais.

In [10]:
resumo_final = (
    df_original_200.groupby(["sample_id", "smell"])
.agg(
        total_avaliacoes=("reviewer_id", "nunique"),
        votos_tem_smell=("tem_smell", "sum"),
        votos_nao_tem_smell=("tem_smell", lambda x: (x == 0).sum())
    )
    .reset_index()
)

def veredito(row):
    if row["votos_tem_smell"] > row["votos_nao_tem_smell"]:
        return 1
    elif row["votos_tem_smell"] < row["votos_nao_tem_smell"]:
        return 0
    else:
        return 0.5  # empate

resumo_final["veredito_final"] = resumo_final.apply(veredito, axis=1)
print(resumo_final.head(5))

   sample_id         smell  total_avaliacoes  votos_tem_smell  \
0    3698323          blob                 1                0   
1    3698323    data class                 1                0   
2    3698602  feature envy                 1                0   
3    3698602   long method                 1                0   
4    3698665  feature envy                 1                0   

   votos_nao_tem_smell  veredito_final  
0                    1             0.0  
1                    1             0.0  
2                    1             0.0  
3                    1             0.0  
4                    1             0.0  


### 6.2 Definição do veredito final por majority vote

O rótulo final de cada instância foi definido com base na regra de majority vote.

Para cada combinação (`sample_id`, `smell`):

- quando o número de votos indicando presença de smell é maior que o número de votos indicando ausência, o rótulo final é definido como presença (1)
- quando o número de votos indicando ausência é maior, o rótulo final é definido como ausência (0)
- em casos de empate, o rótulo é definido como 0.5, indicando indeterminação

In [11]:
resumo_final["veredito_final"].value_counts()

veredito_final
0.0    7767
1.0     807
0.5     137
Name: count, dtype: int64

### 6.3 Incorporação de metadados à base de ground truth

Após a definição do rótulo final, foram incorporadas informações adicionais provenientes da base original, incluindo:

- tipo da entidade (`type`)
- nome do código (`code_name`)
- repositório e commit
- caminho do arquivo (`path`)
- linhas de início e fim
- link para o código
- indicador de relevância industrials.

In [13]:
metadados = (
    df_original_200.groupby(["sample_id", "smell"])
    .agg(
        type=("type", "first"),
        code_name=("code_name", "first"),
        repository=("repository", "first"),
        commit_hash=("commit_hash", "first"),
        path=("path", "first"),
        start_line=("start_line", "first"),
        end_line=("end_line", "first"),
        link=("link", "first"),
        is_from_industry_relevant_project=("is_from_industry_relevant_project", "first")
    )
    .reset_index()
)

### 6.4 Consolidação da base final de ground truth

A base de ground truth foi consolidada por meio da junção entre:

- os resultados agregados das avaliações (majority vote)
- os metadados das instâncias de código

Essa integração resulta em uma base completa, na qual cada linha representa uma instância única (`sample_id`, `smell`) com seu respectivo rótulo final e informações contextuais associadas.

In [14]:
ground_truth_completo = resumo_final.merge(
    metadados,
    on=["sample_id", "smell"],
    how="left"
)

### 6.5 Organização das variáveis da base final

As colunas da base final foram organizadas de forma a facilitar a análise e interpretação dos resultados, incluindo:

- identificação da instância (`sample_id`)
- características do código (`type`, `code_name`, `repository`, etc.)
- informações de localização (`path`, `start_line`, `end_line`)
- resultados das avaliações (`total_avaliacoes`, votos e veredito final)

In [15]:
ground_truth_completo = ground_truth_completo[
    [
        "sample_id",
        "type",
        "smell",
        "code_name",
        "repository",
        "commit_hash",
        "path",
        "start_line",
        "end_line",
        "link",
        "is_from_industry_relevant_project",
        "total_avaliacoes",
        "votos_tem_smell",
        "votos_nao_tem_smell",
        "veredito_final"
    ]
]

### 6.6 Exportação da base completa de ground truth

Salva a base final consolidada de ground truth, contendo os rótulos definidos por majority vote e os metadados associados a cada instância.

In [16]:
ground_truth_completo.to_excel("MLCQ_ground_truth_completo.xlsx", index=False)

### 6.7 Separação da base por tipo de entidade

Separa a base de ground truth em dois subconjuntos distintos, de acordo com o tipo de entidade analisada:

- `class`: instâncias associadas a classes
- `function`: instâncias associadas a métodos/funções

Essa separação é necessária, pois os tipos de code smells avaliados diferem entre classes e funções.

In [20]:
gt_classes = ground_truth_completo[
    ground_truth_completo["type"].astype(str).str.strip().str.lower() == "class"
].copy()

gt_methods = ground_truth_completo[
    ground_truth_completo["type"].astype(str).str.strip().str.lower() == "function"
].copy()

print("Classes:", gt_classes.shape)
print("Methods:", gt_methods.shape)

Classes: (4274, 15)
Methods: (4437, 15)


### 6.8 Exportação das bases por tipo

Exporta as bases separadas de ground truth para classes e funções em arquivos distintos.

In [21]:
gt_classes.to_excel("MLCQ_ground_truth_classes.xlsx", index=False)
gt_methods.to_excel("MLCQ_ground_truth_methods.xlsx", index=False)

### 6.9 Verificação da distribuição por tipo

Apresenta a quantidade de instâncias por tipo (`class` e `function`) na base final

In [23]:
ground_truth_completo["type"].value_counts(dropna=False)

type
function    4437
class       4274
Name: count, dtype: int64

In [24]:
ground_truth_completo["veredito_final"].value_counts()

veredito_final
0.0    7767
1.0     807
0.5     137
Name: count, dtype: int64

In [25]:
mapa_veredito = {
    1: "tem_smell",
    0: "nao_tem_smell",
    0.5: "empate"
}

ground_truth_completo["veredito_final"] = ground_truth_completo["veredito_final"].map(mapa_veredito)
gt_classes["veredito_final"] = gt_classes["veredito_final"].map(mapa_veredito)
gt_methods["veredito_final"] = gt_methods["veredito_final"].map(mapa_veredito)

In [29]:
ground_truth_completo["veredito_final"].value_counts()


veredito_final
nao_tem_smell    7767
tem_smell         807
empate            137
Name: count, dtype: int64

In [30]:
ground_truth_completo.to_excel("MLCQ_ground_truth_completo.xlsx", index=False)
gt_classes.to_excel("MLCQ_ground_truth_classes.xlsx", index=False)
gt_methods.to_excel("MLCQ_ground_truth_methods.xlsx", index=False)

In [ ]:
def repo_to_pasta(repo):
    repo = str(repo).strip()
    repo = repo.replace("git@github.com:", "")
    repo = repo.replace(".git", "")
    owner, nome = repo.split("/")
    return f"{owner}__{nome}"

ground_truth_completo["repo_pasta"] = ground_truth_completo["repository"].apply(repo_to_pasta)

In [ ]:
import os
import pandas as pd

pasta_base = r"C:\Users\Oscar Neto\Desktop\verificar links\downloads"

pastas_locais = set(
    nome for nome in os.listdir(pasta_base)
    if os.path.isdir(os.path.join(pasta_base, nome))
)

pastas_gt = set(ground_truth_completo["repo_pasta"].dropna().unique())

print("Pastas locais:", len(pastas_locais))
print("Pastas no ground truth:", len(pastas_gt))
print("Interseção:", len(pastas_locais & pastas_gt))
print("Só local:", len(pastas_locais - pastas_gt))
print("Só ground truth:", len(pastas_gt - pastas_locais))

In [ ]:
mapa_repos = (
    ground_truth_completo.groupby(["repository", "repo_pasta", "commit_hash"])
    .agg(
        n_smells=("smell", "count"),
        n_arquivos=("path", "nunique")
    )
    .reset_index()
)

mapa_repos["pasta_existe"] = mapa_repos["repo_pasta"].isin(pastas_locais)
mapa_repos.head()

In [ ]:
mapa_repos.to_excel("mapa_repos_sonar.xlsx", index=False)